# Notas — Aula 13: Design Patterns I (Strategy, Command, Factory)

Marco: `EstrategiaZigzag`/`EstrategiaVaivem` provam que Strategy (`mover()`/
`self.estrategia`, desde a Aula 9) aceita qualquer objeto com `mover(robo)`, sem herança.
A fila de comandos vira objetos (`ComandoAvancar`/`ComandoGirar`), guardados em
`robo.historico`, prontos para reexecutar em outro robô. E `criar_robo_configurado`
escolhe classe **e** estratégia a partir de um dicionário de configuração.

In [97]:
from enum import Enum

LADO_GRADE = 10

## Strategy: dando nome a um padrão que já existe

Desde a Aula 6, `Robo.mover()` delega para `self.estrategia.mover(self)` — qualquer
objeto com um método `mover(robo)` serve, sem herdar de nenhuma classe comum. Isso,
com nome, é o **Strategy Pattern**: uma família de algoritmos intercambiáveis, cada um
encapsulado no próprio objeto, escolhida em tempo de execução. No robô, isso é o que
deixa trocar a forma de navegar sem tocar em `Robo`.

In [98]:
class Direcao(Enum):
    LESTE = (1, 0)
    NORTE = (0, 1)
    OESTE = (-1, 0)
    SUL = (0, -1)

    def virar_esquerda(self):
        ordem = [Direcao.LESTE, Direcao.NORTE, Direcao.OESTE, Direcao.SUL]
        return ordem[(ordem.index(self) + 1) % 4]

    def virar_direita(self):
        ordem = [Direcao.LESTE, Direcao.NORTE, Direcao.OESTE, Direcao.SUL]
        return ordem[(ordem.index(self) - 1) % 4]

    def oposta(self):
        dx, dy = self.value
        return Direcao((-dx, -dy))


class EstrategiaPadrao:
    def mover(self, robo):
        return robo.avancar()


class Robo:
    LADO_GRADE = 10
    _registro = {}

    def __init_subclass__(cls, categoria="geral", **kwargs):
        super().__init_subclass__(**kwargs)
        Robo._registro[cls.__name__] = cls
        cls.categoria = categoria

    def __init__(self, nome, x=0, y=0, direcao=Direcao.LESTE, obstaculos=None, estrategia=None):
        self.nome = nome
        self.x = x
        self.y = y
        self.direcao = direcao
        self.obstaculos = obstaculos if obstaculos is not None else {}
        self.estrategia = estrategia if estrategia is not None else EstrategiaPadrao()
        self._historico = []

    @property
    def historico(self):
        return tuple(self._historico)

    def sensor_frente(self):
        dx, dy = self.direcao.value
        nx, ny = self.x + dx, self.y + dy
        return (0 <= nx < Robo.LADO_GRADE and 0 <= ny < Robo.LADO_GRADE
                and (nx, ny) not in self.obstaculos)

    def avancar(self):
        if self.sensor_frente():
            dx, dy = self.direcao.value
            self.x += dx
            self.y += dy
            return True
        return False

    def avancar_n(self, passos):
        for _ in range(passos):
            if not self.avancar():
                break

    def girar(self, lado):
        if lado == "ESQ":
            self.direcao = self.direcao.virar_esquerda()
        elif lado == "DIR":
            self.direcao = self.direcao.virar_direita()

    def mover(self):
        return self.estrategia.mover(self)


print(EstrategiaPadrao.__bases__)

(<class 'object'>,)


## Escrevendo uma nova estratégia: `EstrategiaZigzag`

Nenhuma mudança em `Robo` é necessária — só um objeto novo com `mover(robo)` na
assinatura certa. `EstrategiaZigzag` anda `periodo` passos, vira, anda mais `periodo`
na perpendicular, repete — cobre uma área em faixas, em vez de andar sempre reto.

In [99]:
class EstrategiaZigzag:
    def __init__(self, periodo=2):
        self.periodo = periodo
        self.passos_dados = 0

    def mover(self, robo):
        if self.passos_dados > 0 and self.passos_dados % self.periodo == 0:
            lado = "DIR" if (self.passos_dados // self.periodo) % 2 else "ESQ"
            robo.girar(lado)
        moveu = robo.avancar()
        if moveu:
            self.passos_dados += 1
        return moveu


robo1 = Robo("Wall-E", x=5, y=5, estrategia=EstrategiaZigzag(periodo=2))
for _ in range(4):
    robo1.mover()
print(robo1.x, robo1.y, robo1.direcao.name)

7 3 SUL


### Sua vez

Complete `EstrategiaVaivem`: em vez de **virar 90°** a cada `periodo` passos (como
`EstrategiaZigzag`), ela deve **inverter a direção** — usar `Direcao.oposta()` — fazendo
o robô andar pra frente e pra trás na mesma linha.

*Dica: uma linha, `robo.direcao = robo.direcao.oposta()`, no lugar do `# TODO`.*

In [100]:
class EstrategiaVaivem:
    def __init__(self, periodo=2):
        self.periodo = periodo
        self.passos_dados = 0

    def mover(self, robo):
        if self.passos_dados > 0 and self.passos_dados % self.periodo == 0:
            # TODO: inverta a direção do robô (Direcao.oposta())
            ...
        moveu = robo.avancar()
        if moveu:
            self.passos_dados += 1
        return moveu


robo0 = Robo("Teste", x=5, y=5, estrategia=EstrategiaVaivem(periodo=2))
for _ in range(4):
    robo0.mover()
print(robo0.x, robo0.y, robo0.direcao.name)

9 5 LESTE


## Command: a fila de comandos vira objetos

Desde a D1, comandos são strings (`"AVANCAR 3"`) parseadas na hora. Isso funciona, mas
não dá pra guardar/reexecutar sem reparsear tudo. `Comando` transforma cada instrução num
objeto — com um método `executar(robo)` — pronto para ser guardado, comparado, ou
reexecutado depois, sem depender de string nenhuma.

Repare que `Comando` **não** é `@dataclass`, diferente do `Comando(acao, valor)` da Aula 8.
Aqui cada tipo de comando tem campos próprios (`passos` vs `lado`), e um exercício mais
adiante precisa mutar a instância depois de criada — o que `frozen=True` (o padrão usado em
`Posicao`/`Comando`/`Leitura` na Aula 8) proibiria.


In [101]:
from abc import ABC, abstractmethod


class Comando(ABC):
    @abstractmethod
    def executar(self, robo):
        ...


class ComandoAvancar(Comando):
    def __init__(self, passos):
        self.passos = passos

    def executar(self, robo):
        robo.avancar_n(self.passos)

    def __repr__(self):
        return f"ComandoAvancar({self.passos})"


class ComandoGirar(Comando):
    def __init__(self, lado):
        self.lado = lado

    def executar(self, robo):
        robo.girar(self.lado)

    def __repr__(self):
        return f"ComandoGirar({self.lado!r})"


def parse_comando(texto):
    partes = texto.strip().upper().split()
    acao = partes[0]
    if acao == "AVANCAR":
        return ComandoAvancar(int(partes[1]))
    if acao == "GIRAR":
        return ComandoGirar(partes[1])
    raise ValueError(f"comando desconhecido: {texto!r}")


print(parse_comando("AVANCAR 3"))


ComandoAvancar(3)


### Sua vez

Complete `executar(robo, comandos)`: depois de `cmd.executar(robo)`, falta **guardar**
o comando executado em `robo._historico`, para virar `robo.historico` reexecutável.

*Dica: `robo._historico.append(cmd)`.*

In [102]:
def executar(robo, comandos):
    for texto in comandos:
        cmd = parse_comando(texto)
        cmd.executar(robo)
        # TODO: guarde cmd em robo._historico
        robo._historico.append(cmd)
        ...


robo2 = Robo("Wall-E")
executar(robo2, ["GIRAR ESQ", "AVANCAR 2"])
print(robo2.x, robo2.y, robo2.historico)

0 2 (ComandoGirar('ESQ'), ComandoAvancar(2))


## O que ganhamos: reexecutar o histórico em outro robô

Porque `historico` guarda **objetos**, não texto, dá pra reexecutar a mesma sequência
em outro robô sem reparsear nada — só iterar e chamar `executar` de novo.

In [103]:
robo3 = Robo("Bender")
for cmd in robo2.historico:
    cmd.executar(robo3)
print(robo3.x, robo3.y, robo3.direcao)

0 2 Direcao.NORTE


## Factory: escolhendo a classe por `Robo._registro`

Desde a Aula 8, toda subclasse de `Robo` se registra sozinha em `Robo._registro` via
`__init_subclass__`. `criar_robo` usa esse catálogo para decidir qual classe instanciar
a partir só do **nome** — sem `if/elif` gigante, sem `RoboVeloz(...)` escrito à mão em
quem chama.

In [104]:
class RoboVeloz(Robo, categoria="ofensivo"):
    pass


class RoboExplorador(Robo, categoria="reconhecimento"):
    pass


def criar_robo(tipo_nome, nome, **kwargs):
    classe = Robo._registro.get(tipo_nome)
    if classe is None:
        raise ValueError(f"tipo desconhecido: {tipo_nome!r}")
    return classe(nome, **kwargs)


r = criar_robo("RoboVeloz", "Speedy")
print(r.nome, r.categoria)

Speedy ofensivo


### Sua vez

Complete `criar_robo_configurado`: depois de criar o robô com `criar_robo`, falta
**trocar** `robo.estrategia` pela classe de estratégia escolhida (já instanciada).

*Dica: `robo.estrategia = classe_estrategia()`.*

In [105]:
FABRICA_ESTRATEGIAS = {
    "padrao": EstrategiaPadrao,
    "zigzag": EstrategiaZigzag,
    "vaivem": EstrategiaVaivem,
}


def criar_robo_configurado(tipo_nome, nome, estrategia_nome="padrao", **kwargs):
    classe_estrategia = FABRICA_ESTRATEGIAS.get(estrategia_nome)
    if classe_estrategia is None:
        raise ValueError(f"estrategia desconhecida: {estrategia_nome!r}")
    robo = criar_robo(tipo_nome, nome, **kwargs)
    # TODO: troque robo.estrategia pela classe_estrategia recém escolhida
    robo.estrategia = classe_estrategia()
    ...
    return robo


r2 = criar_robo_configurado("RoboExplorador", "Scout", estrategia_nome="zigzag")
print(type(r2.estrategia).__name__)
print(EstrategiaZigzag)



EstrategiaZigzag
<class '__main__.EstrategiaZigzag'>


## Para aprofundar

- Strategy Pattern — Refactoring.Guru: https://refactoring.guru/design-patterns/strategy
- Duck typing — glossário oficial: https://docs.python.org/3/glossary.html#term-duck-typing
- Command Pattern — Refactoring.Guru: https://refactoring.guru/design-patterns/command
- Factory Method / Simple Factory — Refactoring.Guru: https://refactoring.guru/design-patterns/factory-method
- Singleton — mecânica e críticas — Real Python: https://realpython.com/python-singleton/